# POS Terminal — AIMternet-Cafe front desk

This is the till. Run the cells top to bottom to serve a customer.

**How this works.** Everything on this page goes through the cafe's API over HTTP. The
notebook holds no passwords, opens no database connection, and works out no prices of its
own — it asks the server and shows you the answer. That is deliberate: if the till could
compute its own prices, two tills could disagree, and neither would be wrong on paper.

So when you see a total below, it is the total the cafe's own rules produced.

**Before you start:** the API must be running (`make api`). The next cell checks.

In [ ]:
import requests   # the only thing this notebook needs: an HTTP client

API = "http://127.0.0.1:8000"


def ask(method, path, body=None, key=None):
    """Send one request to the cafe API and hand back the answer.

    If something is refused, we print the reason the server gave rather than a stack trace —
    "PC-014 is occupied" is what the front desk needs to see.
    """
    headers = {"Idempotency-Key": key} if key else {}
    response = requests.request(method, API + path, json=body, headers=headers, timeout=30)
    answer = response.json() if response.content else {}
    if not response.ok:
        print(f"  Refused ({response.status_code}): {answer.get('detail', answer)}")
        return None
    return answer


health = ask("GET", "/healthz")
if health and health["status"] == "ok":
    print(f"Connected to the cafe API. Database: {health['database']}.")
else:
    print("The API is not answering. Start it with:  make api")

## 1. Which computers are free?

The zone decides the hourly rate: Standard ₱50, VIP Esports ₱80, Streamer Pods ₱120.

In [ ]:
free = ask("GET", "/v1/workstations/available")

print(f"{free['count']} computers are free right now.\n")
by_zone = {}
for pc in free["workstations"]:
    by_zone.setdefault(pc["zone_classification"], []).append(pc)

for zone, machines in by_zone.items():
    rate = machines[0]["base_hourly_rate"]
    names = ", ".join(m["workstation_id"] for m in machines[:12])
    more = f" … and {len(machines) - 12} more" if len(machines) > 12 else ""
    print(f"{zone}  —  PHP {rate}/hour  —  {len(machines)} free")
    print(f"   {names}{more}\n")

## 2. Look up the customer

Type the member's card number below. You will see their tier, their points, and whether they
are already sitting at a machine.

In [ ]:
MEMBER_ID = "M-1841"      # <-- the customer's card number

member = ask("GET", f"/v1/members/{MEMBER_ID}")

if member:
    print(f"{member['first_name']} {member['last_name']}")
    print(f"  Tier            : {member['current_tier']}")
    print(f"  Points          : {member['current_points_balance']}")
    print(f"  Can redeem      : {member['redeemable_units']} x 100 points "
          f"(worth PHP {member['redeemable_units'] * 50}.00 off a rental)")
    print(f"  Lifetime spend  : PHP {member['lifetime_spend_amount']}")
    if member["open_rental_id"]:
        print(f"  ALREADY SEATED  : rental {member['open_rental_id']} is still open")
    else:
        print("  Not currently seated — ready to check in.")

## 3. Check them in

Pick a free computer and how many hours they are paying for. The server prices it: it applies
their tier discount and records the session.

A customer can only be at one machine at a time, and a machine can only hold one customer —
the server enforces both, so you cannot double-book by accident.

In [ ]:
WORKSTATION_ID = free["workstations"][0]["workstation_id"]   # or type one, e.g. "PC-014"
HOURS = "3.00"

rental = ask("POST", "/v1/rentals/check-in",
             {"member_id": MEMBER_ID, "workstation_id": WORKSTATION_ID, "duration_hours": HOURS})

if rental:
    print(f"Checked in: {rental['member_id']} -> {rental['workstation_id']} "
          f"({rental['zone_classification']})")
    print(f"  Receipt no.   : {rental['rental_id']}")
    print(f"  Standard rate : PHP {rental['base_hourly_rate']}/hour")
    print(f"  Their rate    : PHP {rental['final_hourly_rate']}/hour "
          f"({member['current_tier']} discount {float(rental['tier_discount_pct']) * 100:.0f}%)")
    print(f"  Booked        : {rental['duration_hours']} hours")

## 4. Sell them something

Send the item codes and how many. **You never type a price** — the server looks up what each
item costs, checks there is stock, works out the total and the points earned, and takes the
stock off the shelf. All of it either happens together or not at all.

In [ ]:
menu = ask("GET", "/v1/concessions/catalog")

print(f"{'code':14s} {'item':34s} {'price':>9s} {'in stock':>9s}")
print("-" * 70)
for item in menu:
    print(f"{item['item_sku']:14s} {item['item_name']:34s} "
          f"{'PHP ' + str(item['unit_retail_price']):>9s} {item['stock_quantity']:>9d}")

In [ ]:
ORDER = [
    {"item_sku": "SKU-BEV-01", "quantity": 2},   # Iced Mountain Tea
    {"item_sku": "SKU-SNK-01", "quantity": 1},   # Spicy Tonkotsu Ramen Bowl
]

purchase = ask("POST", "/v1/concessions/purchases",
               {"member_id": MEMBER_ID,
                "rental_id": rental["rental_id"] if rental else None,
                "payment_method": "GCash",
                "items": ORDER})

if purchase:
    print(f"Order {purchase['purchase_id']}\n")
    for line in purchase["items"]:
        print(f"  {line['quantity']} x {line['item_name']:34s} "
              f"PHP {line['unit_price']:>8s}   =  PHP {line['total_price']}")
    print(f"\n  {'TOTAL':>44s}      PHP {purchase['total_amount']}")
    print(f"  {'Points earned':>44s}      {purchase['points_accrued']}")

## 5. Check them out

This closes the session, works out the final bill, frees the computer and credits their
points.

If they want to spend points, set `REDEEM` to a multiple of 100. Each 100 points takes ₱50
off. If they do not have enough, the server says so and nothing is charged — set it back to 0
and run the cell again.

In [ ]:
REDEEM = 0          # e.g. 100 or 200. Must be a multiple of 100.

closed = ask("POST", "/v1/rentals/check-out",
             {"rental_id": rental["rental_id"] if rental else None,
              "points_to_redeem": REDEEM,
              "payment_method": "Cash"})

if closed:
    print(f"Receipt {closed['rental_id']}\n")
    print(f"  Computer        : {closed['workstation_id']} ({closed['zone_classification']})")
    print(f"  Time used       : {closed['duration_hours']} hours")
    print(f"  Rate            : PHP {closed['final_hourly_rate']}/hour")
    print(f"  Subtotal        : PHP {closed['gross_rental_amount']}")
    if int(closed["points_redeemed"]):
        print(f"  Points redeemed : {closed['points_redeemed']} "
              f"(-PHP {closed['points_credit_value']})")
    print(f"  TO PAY          : PHP {closed['net_amount_paid']}  ({closed['payment_method']})")
    print(f"  Points earned   : {closed['points_accrued']}")

In [ ]:
after = ask("GET", f"/v1/members/{MEMBER_ID}")
if after and member:
    print(f"{after['first_name']} {after['last_name']} — {after['current_tier']}")
    print(f"  Points  : {member['current_points_balance']}  ->  {after['current_points_balance']}")
    print(f"  Spend   : PHP {member['lifetime_spend_amount']}  ->  PHP {after['lifetime_spend_amount']}")
    if after["current_tier"] != member["current_tier"]:
        print(f"  UPGRADED to {after['current_tier']}!")

## 6. What happens when something is wrong

Worth seeing once, so the messages are familiar when they appear for real. None of these
change anything — the server refuses before it writes.

In [ ]:
print("Seating someone at a computer that is already taken:")
# Seat someone briefly so the refusal below is real rather than hypothetical.
spare = ask("GET", "/v1/workstations/available")["workstations"][0]["workstation_id"]
held = ask("POST", "/v1/rentals/check-in",
           {"member_id": "M-1842", "workstation_id": spare, "duration_hours": "1.00"})
if held:
    print(f"  (M-1842 is now at {spare}. Trying to seat someone else there…)")
    ask("POST", "/v1/rentals/check-in",
        {"member_id": "M-1843", "workstation_id": spare, "duration_hours": "1.00"})

print("\nSeating a customer who is already at another computer:")
other = ask("GET", "/v1/workstations/available")["workstations"][0]["workstation_id"]
ask("POST", "/v1/rentals/check-in",
    {"member_id": "M-1842", "workstation_id": other, "duration_hours": "1.00"})

print("\nSpending more points than the customer has:")
ask("POST", "/v1/rentals/check-out",
    {"rental_id": held["rental_id"] if held else "", "points_to_redeem": 100000})

print("\nSpending points in an amount that is not a multiple of 100:")
ask("POST", "/v1/rentals/check-out",
    {"rental_id": held["rental_id"] if held else "", "points_to_redeem": 150})

# Tidy up the demonstration seat.
if held:
    ask("POST", "/v1/rentals/check-out", {"rental_id": held["rental_id"]})

print("\nA card number that does not exist:")
ask("GET", "/v1/members/M-9999")

print("\nAn item code that does not exist:")
ask("POST", "/v1/concessions/purchases",
    {"member_id": MEMBER_ID, "items": [{"item_sku": "SKU-XXX-99", "quantity": 1}]})

print("\nOrdering more stock than the shelf holds:")
scarce = min(menu, key=lambda i: i["stock_quantity"])
ask("POST", "/v1/concessions/purchases",
    {"member_id": MEMBER_ID,
     "items": [{"item_sku": scarce["item_sku"], "quantity": scarce["stock_quantity"] + 1}]})

print("\nClosing a session that is already closed:")
if closed:
    ask("POST", "/v1/rentals/check-out", {"rental_id": closed["rental_id"]})


---

### For whoever maintains this

This notebook imports `requests` and nothing else. There is no `psycopg2`, no `boto3`, no
`sqlalchemy`, no SQL and no credentials anywhere in it, and
`tests/unit/test_notebook_boundary.py` fails the build if any of those ever appear.

That boundary is the point. Prices, tier discounts, point maths, stock levels and the rule
that one person cannot occupy two machines all live in the API, in one place, where they can
be tested. The till just asks.